<div dir="rtl">

# نوت‌بوک ۵: تشخیص ایمیل اسپم با Naive Bayes و استخراج کلمات مهم هر کلاس

## مقدمه
فیلتر اسپم یکی از اولین کاربردهای موفق یادگیری ماشین بود (Paul Graham، ۲۰۰۲) و الگوریتم **Naive Bayes (بیز ساده)** هنوز هم یکی از بهترین نقاط شروع برای **طبقه‌بندی متن** است: سریع، ساده، با داده کم هم خوب کار می‌کند و **قابل تفسیر** است.

### دیتاست: Enron-Spam
مجموعه‌ای از حدود **۳۳۰۰۰ ایمیل واقعی** شامل ایمیل‌های سالم (Ham) کارمندان شرکت Enron و ایمیل‌های اسپم (Metsis و همکاران، ۲۰۰۶). نسخه تمیزشده آن از مخزن GitHub دانلود می‌شود (حدود ۱۵ مگابایت) و یک بار در پوشه `data/` ذخیره می‌شود. اگر دانلود ممکن نبود، نوت‌بوک به‌طور خودکار از دیتاست **SMS Spam Collection** استفاده می‌کند.

### فهرست مطالب
1. تئوری Naive Bayes
2. بارگذاری داده
3. تحلیل اکتشافی داده (EDA)
4. تبدیل متن به عدد: Bag-of-Words
5. آموزش MultinomialNB
6. ارزیابی: ماتریس درهم‌ریختگی، Precision، Recall
7. منحنی‌های ROC و Precision-Recall و انتخاب آستانه
8. مقایسه انواع Naive Bayes با Cross Validation
9. تنظیم ابرپارامترها با GridSearchCV
10. **کلمات مهم هر کلاس**
11. توضیح پیش‌بینی برای یک ایمیل خاص
12. تحلیل خطا
13. آزمایش با ایمیل‌های دلخواه
14. جمع‌بندی

</div>

<div dir="rtl">

_## ۱. تئوری Naive Bayes

### قضیه بیز
$$P(\text{spam} \mid \text{email}) = \frac{P(\text{email} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{email})}$$

- $P(\text{spam})$: احتمال **پیشین (Prior)** — نسبت ایمیل‌های اسپم در داده آموزش
- $P(\text{email} \mid \text{spam})$: **درست‌نمایی (Likelihood)** — احتمال دیدن این ایمیل اگر اسپم باشد
- $P(\text{spam} \mid \text{email})$: احتمال **پسین (Posterior)** — چیزی که می‌خواهیم

### فرض «ساده» (Naive)
محاسبه $P(\text{email} \mid \text{spam})$ برای کل متن غیرممکن است. Naive Bayes فرض می‌کند **کلمات به شرط کلاس، مستقل از هم هستند**:
$$P(w_1, w_2, \dots, w_n \mid c) \approx \prod_{i=1}^{n} P(w_i \mid c)$$
این فرض در واقعیت غلط است (مثلاً «free» و «money» با هم می‌آیند)، اما در عمل طبقه‌بندی بسیار خوبی می‌دهد چون برای تصمیم فقط **ترتیب** احتمالات مهم است، نه مقدار دقیق آن‌ها.

### Multinomial Naive Bayes (مناسب شمارش کلمات)
$$\hat{c} = \arg\max_c \Big[\log P(c) + \sum_{w} x_w \log P(w \mid c)\Big]$$
که $x_w$ تعداد تکرار کلمه $w$ در ایمیل است. احتمال هر کلمه در هر کلاس با **هموارسازی لاپلاس (Laplace/Additive Smoothing)** تخمین زده می‌شود:
$$P(w \mid c) = \frac{N_{c,w} + \alpha}{N_c + \alpha \lvert V \rvert}$$
- $N_{c,w}$: تعداد تکرار کلمه $w$ در همه ایمیل‌های کلاس $c$
- $N_c$: تعداد کل کلمات کلاس $c$، $\lvert V \rvert$: اندازه واژگان
- $\alpha$: پارامتر هموارسازی (پیش‌فرض ۱). **چرا لازم است؟** بدون آن، اگر کلمه‌ای در داده آموزش اسپم نیامده باشد، $P(w \mid \text{spam}) = 0$ می‌شود و کل حاصل‌ضرب صفر می‌شود!

**چرا لگاریتم؟** ضرب صدها احتمال کوچک باعث **Underflow** عددی می‌شود؛ جمع لگاریتم‌ها پایدار است.

### انواع Naive Bayes در scikit-learn
| مدل | نوع ویژگی | کاربرد |
|---|---|---|
| `MultinomialNB` | شمارش (Count) یا TF-IDF | **طبقه‌بندی متن** (رایج‌ترین) |
| `ComplementNB` | شمارش / TF-IDF | متن با **کلاس‌های نامتوازن** |
| `BernoulliNB` | دودویی (کلمه هست/نیست) | متن‌های کوتاه؛ نبود کلمه را هم در نظر می‌گیرد |
| `GaussianNB` | پیوسته با توزیع نرمال | داده‌های عددی پیوسته (نه متن) |
| `CategoricalNB` | دسته‌ای | ویژگی‌های گسسته غیرمتنی |

</div>_

In [ ]:
import re
import time
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, classification_report,
                             ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.rcParams.update({'figure.dpi': 100})

<div dir="rtl">

## ۲. بارگذاری داده

- ایمیل‌ها یک بار دانلود و در `data/enron_spam_data.zip` ذخیره می‌شوند.
- متن نهایی هر ایمیل = **موضوع (Subject) + بدنه (Message)**.
- برچسب: `1` = اسپم، `0` = سالم (Ham).
- **حذف ایمیل‌های تکراری:** اگر یک ایمیل هم در آموزش و هم در آزمون باشد، دقت آزمون به صورت کاذب بالا می‌رود (**نشت داده / Data Leakage**). اسپم‌ها معمولاً بارها با متن یکسان ارسال می‌شوند، پس این مرحله مهم است.

</div>

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
ENRON_URL = 'https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip'
SMS_URL = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'


def load_spam_data():
    enron_path = DATA_DIR / 'enron_spam_data.zip'
    try:
        if not enron_path.exists():
            print('Downloading Enron-Spam dataset ...')
            urllib.request.urlretrieve(ENRON_URL, enron_path)
        raw = pd.read_csv(enron_path, compression='zip')
        df = pd.DataFrame({'text': raw['Subject'].fillna('') + ' ' + raw['Message'].fillna(''),
                           'label': (raw['Spam/Ham'] == 'spam').astype(int)})
        return df, 'Enron-Spam (emails)'
    except Exception as e:                                   # fallback: SMS Spam Collection
        print('Enron download failed -> using SMS Spam Collection instead:', e)
        raw = pd.read_csv(SMS_URL, sep='\t', header=None, names=['label', 'text'])
        return pd.DataFrame({'text': raw['text'], 'label': (raw['label'] == 'spam').astype(int)}), 'SMS Spam Collection'


df, source = load_spam_data()
print(f'Source: {source} | raw rows: {len(df)}')
df['text'] = df['text'].str.strip()
df = df[df['text'].str.len() > 0]
n_before = len(df)
df = df.drop_duplicates(subset='text').reset_index(drop=True)
print(f'Removed {n_before - len(df)} duplicate / empty emails -> {len(df)} unique emails')
df.head()

<div dir="rtl">

## ۳. تحلیل اکتشافی داده (EDA)

### ۳-۱. توازن کلاس‌ها و طول ایمیل‌ها

</div>

In [ ]:
df['n_words'] = df['text'].str.split().str.len()
counts = df['label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [1, 2]})
bars = axes[0].bar(['ham (0)', 'spam (1)'], counts.values, color=['tab:green', 'tab:red'])
axes[0].bar_label(bars, labels=[f'{v}\n({v / counts.sum():.1%})' for v in counts.values])
axes[0].set_title('Class balance'); axes[0].set_ylabel('number of emails')
axes[0].set_ylim(0, counts.max() * 1.2)

bins = np.logspace(0, np.log10(df['n_words'].max() + 1), 50)
for lab, name, col in [(0, 'ham', 'tab:green'), (1, 'spam', 'tab:red')]:
    axes[1].hist(df.loc[df.label == lab, 'n_words'], bins=bins, alpha=0.55, label=name, color=col)
axes[1].set_xscale('log'); axes[1].set_xlabel('number of words (log scale)'); axes[1].set_ylabel('count')
axes[1].set_title('Email length distribution'); axes[1].legend()
plt.tight_layout()
plt.show()

display(df.groupby('label')['n_words'].describe().round(1))

<div dir="rtl">

### ۳-۲. چند نمونه از هر کلاس

</div>

In [ ]:
for lab, name in [(0, 'HAM'), (1, 'SPAM')]:
    print(f'================ {name} ================')
    for t in df[df.label == lab].sample(3, random_state=1)['text']:
        print('-', re.sub(r'\s+', ' ', t)[:300], '...\n')

<div dir="rtl">

## ۴. تبدیل متن به عدد: مدل کیسه کلمات (Bag-of-Words)

مدل‌های یادگیری ماشین فقط با عدد کار می‌کنند. در **Bag-of-Words**:
1. **Tokenization:** متن به کلمات شکسته می‌شود.
2. **ساخت واژگان (Vocabulary):** فهرست همه کلمات یکتا.
3. **برداری کردن:** هر سند یک بردار به طول واژگان است که هر درایه آن **تعداد تکرار** یک کلمه است.

ترتیب کلمات نادیده گرفته می‌شود (مثل ریختن کلمات در یک کیسه!). ماتریس حاصل **بسیار اسپارس** است، چون هر ایمیل فقط تعداد کمی از کل کلمات را دارد.

### مثال کوچک

</div>

In [ ]:
toy_docs = ['Win free money now', 'Meeting schedule for the project now', 'Free free offer, claim your money']
toy_vec = CountVectorizer(lowercase=True)
toy_X = toy_vec.fit_transform(toy_docs)
pd.DataFrame(toy_X.toarray(), columns=toy_vec.get_feature_names_out(), index=[f'doc {i + 1}' for i in range(3)])

<div dir="rtl">

### تنظیمات `CountVectorizer` در این نوت‌بوک
| پارامتر | مقدار | دلیل |
|---|---|---|
| `lowercase` | `True` | «Free» و «free» یک کلمه‌اند |
| `stop_words` | `'english'` | حذف کلمات پرتکرار بی‌معنا (the، and، is، ...) |
| `token_pattern` | فقط حروف، حداقل ۲ حرف | حذف اعداد و کدهای بی‌معنا |
| `min_df` | `3` | حذف کلماتی که در کمتر از ۳ ایمیل آمده‌اند (نویز، غلط املایی) |
| `max_df` | `0.9` | حذف کلماتی که در بیش از ۹۰٪ ایمیل‌ها هستند |

### تقسیم داده
۸۰٪ آموزش و ۲۰٪ آزمون، به صورت **Stratified** تا نسبت اسپم در هر دو یکسان باشد.

</div>

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['text'].values, df['label'].values, test_size=0.2, stratify=df['label'], random_state=RANDOM_STATE)
print(f'Train: {len(X_train_txt)} emails | Test: {len(X_test_txt)} emails | spam ratio train={y_train.mean():.3f}, test={y_test.mean():.3f}')

TOKEN_PATTERN = r'(?u)\b[a-zA-Z][a-zA-Z]+\b'


def make_count_vectorizer(**kw):
    params = dict(lowercase=True, stop_words='english', token_pattern=TOKEN_PATTERN, min_df=3, max_df=0.9)
    params.update(kw)
    return CountVectorizer(**params)

<div dir="rtl">

## ۵. آموزش MultinomialNB

از `Pipeline` استفاده می‌کنیم تا برداری‌سازی و مدل یک واحد باشند. مزیت مهم: در Cross Validation، واژگان فقط از داده آموزش هر Fold ساخته می‌شود و **نشت داده** رخ نمی‌دهد.

</div>

In [ ]:
nb_pipe = Pipeline([('vect', make_count_vectorizer()), ('clf', MultinomialNB(alpha=1.0))])

t0 = time.time()
nb_pipe.fit(X_train_txt, y_train)
print(f'Training time: {time.time() - t0:.2f} s')

vect, clf = nb_pipe.named_steps['vect'], nb_pipe.named_steps['clf']
vocab = vect.get_feature_names_out()
print(f'Vocabulary size: {len(vocab):,} words')
print(f'Class priors P(ham), P(spam): {np.exp(clf.class_log_prior_).round(4)}')
print(f'feature_log_prob_ shape (classes x words): {clf.feature_log_prob_.shape}')

<div dir="rtl">

## ۶. ارزیابی مدل

در فیلتر اسپم دو نوع خطا داریم که **هزینه یکسانی ندارند**:
- **False Positive (FP):** ایمیل سالم به پوشه اسپم برود ← **خیلی بد!** (مثلاً ایمیل پیشنهاد کاری گم شود)
- **False Negative (FN):** اسپم وارد صندوق ورودی شود ← آزاردهنده ولی کم‌خطرتر

بنابراین برای کلاس اسپم، **Precision** (از ایمیل‌هایی که اسپم تشخیص داده‌ایم، چند درصد واقعاً اسپم‌اند) اهمیت ویژه‌ای دارد.

</div>

In [ ]:
y_pred = nb_pipe.predict(X_test_txt)
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}   (spam class)')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}   (spam class)')
print(f'F1-score : {f1_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred, target_names=['ham', 'spam'], digits=4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['ham', 'spam'], ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion matrix (counts)')
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['ham', 'spam'], ax=axes[1], cmap='Blues',
                                        colorbar=False, normalize='true', values_format='.3%')
axes[1].set_title('Confusion matrix (row-normalized)')
plt.tight_layout()
plt.show()
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f'Ham emails wrongly sent to spam (FP): {fp}  |  spam emails that reached the inbox (FN): {fn}')

<div dir="rtl">

## ۷. منحنی‌های ROC و Precision-Recall و انتخاب آستانه

### ۷-۱. مشکل احتمالات Naive Bayes
به دلیل فرض استقلال، Naive Bayes شواهد کلمات هم‌بسته را چند بار حساب می‌کند و احتمالاتش **بیش از حد مطمئن** (نزدیک ۰ یا ۱) می‌شوند؛ یعنی **کالیبره نیست**. هیستوگرام زیر این را نشان می‌دهد.

به همین دلیل برای رسم منحنی‌ها و انتخاب آستانه، به جای احتمال از **لگاریتم نسبت شانس (Log-odds)** استفاده می‌کنیم:
$$s(x) = \log P(\text{spam} \mid x) - \log P(\text{ham} \mid x)$$
که ترتیب نمونه‌ها را دقیق‌تر حفظ می‌کند (آستانه پیش‌فرض ۰.۵ روی احتمال معادل $s = 0$ است).

</div>

In [ ]:
proba_test = nb_pipe.predict_proba(X_test_txt)[:, 1]
log_p = nb_pipe.predict_log_proba(X_test_txt)
score_test = log_p[:, 1] - log_p[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].hist(proba_test[y_test == 0], bins=50, alpha=0.6, color='tab:green', label='ham')
axes[0].hist(proba_test[y_test == 1], bins=50, alpha=0.6, color='tab:red', label='spam')
axes[0].set_yscale('log'); axes[0].set_xlabel('P(spam | email)'); axes[0].set_ylabel('count (log)')
axes[0].set_title('Predicted probabilities are pushed to 0 or 1'); axes[0].legend()

clip = np.clip(score_test, -200, 200)
axes[1].hist(clip[y_test == 0], bins=80, alpha=0.6, color='tab:green', label='ham')
axes[1].hist(clip[y_test == 1], bins=80, alpha=0.6, color='tab:red', label='spam')
axes[1].axvline(0, color='k', ls='--', label='default threshold (s = 0)')
axes[1].set_xlabel('log-odds score s(x)  (clipped to [-200, 200])'); axes[1].set_ylabel('count')
axes[1].set_title('Log-odds scores separate the classes'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, score_test)
prec, rec, _ = precision_recall_curve(y_test, score_test)
auc_val, ap_val = roc_auc_score(y_test, score_test), average_precision_score(y_test, score_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot(fpr, tpr, lw=2, label=f'MultinomialNB (AUC = {auc_val:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='random')
axes[0].set_xlabel('False Positive Rate (ham -> spam)'); axes[0].set_ylabel('True Positive Rate (spam recall)')
axes[0].set_title('ROC curve'); axes[0].legend(loc='lower right')
ins = axes[0].inset_axes([0.35, 0.3, 0.4, 0.4])
ins.plot(fpr, tpr, lw=2); ins.set_xlim(0, 0.05); ins.set_ylim(0.85, 1.0); ins.set_title('zoom', fontsize=8)

axes[1].plot(rec, prec, lw=2, color='tab:purple', label=f'MultinomialNB (AP = {ap_val:.4f})')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'random (spam ratio = {y_test.mean():.2f})')
axes[1].set_xlabel('Recall (spam)'); axes[1].set_ylabel('Precision (spam)')
axes[1].set_title('Precision-Recall curve'); axes[1].legend(loc='lower left')
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۷-۲. انتخاب آستانه برای کاهش ایمیل‌های سالمِ از دست رفته
هدف: **نرخ FP (ایمیل سالم در پوشه اسپم) حداکثر ۰.۵٪** باشد.

**روش درست:** آستانه را **نباید** با داده آزمون انتخاب کنیم. با `cross_val_predict` برای هر ایمیل آموزشی یک امتیاز «خارج از نمونه» (out-of-fold) می‌گیریم، آستانه را روی آن تنظیم می‌کنیم و سپس فقط یک بار روی داده آزمون ارزیابی می‌کنیم.

</div>

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_log_p = cross_val_predict(nb_pipe, X_train_txt, y_train, cv=cv5, method='predict_log_proba', n_jobs=-1)
oof_score = oof_log_p[:, 1] - oof_log_p[:, 0]

TARGET_FPR = 0.005
fpr_tr, tpr_tr, thr_tr = roc_curve(y_train, oof_score)
ok = fpr_tr <= TARGET_FPR
best_thr = thr_tr[ok][np.argmax(tpr_tr[ok])]
print(f'Chosen log-odds threshold (from out-of-fold training scores): {best_thr:.2f}')

rows = []
for name, t in [('default (s > 0)', 0.0), (f'tuned (FPR <= {TARGET_FPR:.1%})', best_thr)]:
    pred = (score_test > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({'threshold': name, 'precision': precision_score(y_test, pred), 'recall': recall_score(y_test, pred),
                 'F1': f1_score(y_test, pred), 'FP (ham lost)': fp, 'FN (spam missed)': fn, 'FPR': fp / (fp + tn)})
display(pd.DataFrame(rows).set_index('threshold').round(4))

<div dir="rtl">

**تفسیر:** با بالا بردن آستانه، تعداد ایمیل‌های سالمی که به اشتباه اسپم تشخیص داده می‌شوند کم می‌شود و در عوض کمی اسپم بیشتر وارد صندوق ورودی می‌شود. این همان **مصالحه Precision/Recall** است که با منحنی‌ها دیدیم؛ انتخاب نقطه مناسب یک **تصمیم کسب‌وکاری** است، نه صرفاً فنی.

</div>

<div dir="rtl">

## ۸. مقایسه انواع Naive Bayes با Cross Validation

پنج پیکربندی را با **5-Fold Cross Validation** روی داده آموزش مقایسه می‌کنیم:
1. `MultinomialNB` + شمارش کلمات (Count)
2. `MultinomialNB` + **TF-IDF**: وزن کلمات رایج در همه اسناد را کم می‌کند: $\text{tfidf}(w,d) = \text{tf}(w,d) \cdot \log\frac{1+N}{1+\text{df}(w)} + 1$
3. `ComplementNB` + TF-IDF: پارامترها را از **مکمل** هر کلاس تخمین می‌زند؛ برای داده نامتوازن پایدارتر است
4. `BernoulliNB` + ویژگی دودویی: فقط وجود/عدم وجود کلمه
5. `LogisticRegression` + TF-IDF: یک مدل خطی **تمایزی (Discriminative)** به عنوان مرجع مقایسه

> **مدل مولد در مقابل تمایزی:** Naive Bayes یک مدل **مولد (Generative)** است ($P(x \mid y)$ را مدل می‌کند)، در حالی که رگرسیون لجستیک **تمایزی** است ($P(y \mid x)$ را مستقیم مدل می‌کند). معمولاً مدل‌های مولد با داده کم سریع‌تر به عملکرد خوب می‌رسند و مدل‌های تمایزی با داده زیاد دقیق‌ترند (Ng & Jordan, 2002).

</div>

In [ ]:
def make_tfidf(**kw):
    params = dict(lowercase=True, stop_words='english', token_pattern=TOKEN_PATTERN, min_df=3, max_df=0.9, sublinear_tf=True)
    params.update(kw)
    return TfidfVectorizer(**params)


models = {
    'MultinomialNB + Count': Pipeline([('vect', make_count_vectorizer()), ('clf', MultinomialNB())]),
    'MultinomialNB + TF-IDF': Pipeline([('vect', make_tfidf()), ('clf', MultinomialNB())]),
    'ComplementNB + TF-IDF': Pipeline([('vect', make_tfidf()), ('clf', ComplementNB())]),
    'BernoulliNB + Binary': Pipeline([('vect', make_count_vectorizer(binary=True, max_features=30000)), ('clf', BernoulliNB())]),
    'LogisticRegression + TF-IDF': Pipeline([('vect', make_tfidf()), ('clf', LogisticRegression(C=10, max_iter=2000))]),
}
scoring = ['accuracy', 'precision', 'recall', 'f1']
cv_results = {}
t0 = time.time()
for name, model in models.items():
    cv_results[name] = cross_validate(model, X_train_txt, y_train, cv=cv5, scoring=scoring, n_jobs=-1)
print(f'Cross-validation done in {time.time() - t0:.1f} s')

summary = pd.DataFrame({name: {**{m: r[f'test_{m}'].mean() for m in scoring},
                               'f1_std': r['test_f1'].std(), 'fit_time_s': r['fit_time'].mean()}
                        for name, r in cv_results.items()}).T.sort_values('f1', ascending=False)
display(summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={'width_ratios': [1.4, 1]})
names = list(cv_results.keys())
axes[0].boxplot([cv_results[n]['test_f1'] for n in names], widths=0.5, showmeans=True)
axes[0].set_xticks(range(1, len(names) + 1))
axes[0].set_xticklabels([n.replace(' + ', '\n+ ') for n in names], fontsize=9)
axes[0].set_ylabel('F1-score (spam)'); axes[0].set_title('5-fold CV F1-score distribution'); axes[0].grid(alpha=0.3)

x = np.arange(len(summary)); w = 0.2
for i, m in enumerate(['accuracy', 'precision', 'recall', 'f1']):
    axes[1].bar(x + (i - 1.5) * w, summary[m], w, label=m)
axes[1].set_xticks(x); axes[1].set_xticklabels([n.replace(' + ', '\n+ ') for n in summary.index], fontsize=8)
axes[1].set_ylim(0.85, 1.0); axes[1].legend(fontsize=8, loc='lower left'); axes[1].set_title('Mean CV metrics')
plt.tight_layout()
plt.show()

<div dir="rtl">

## ۹. تنظیم ابرپارامترها با `GridSearchCV`

دو ابرپارامتر مهم:
- `clf__alpha`: میزان هموارسازی لاپلاس. $\alpha$ کوچک ← اعتماد بیشتر به شمارش‌های مشاهده‌شده (خطر بیش‌برازش روی کلمات نادر)؛ $\alpha$ بزرگ ← احتمالات به سمت یکنواخت می‌روند (کم‌برازش).
- `vect__ngram_range`: `(1, 1)` فقط تک‌کلمه‌ها؛ `(1, 2)` تک‌کلمه‌ها + **جفت‌کلمه‌ها (Bigrams)** مثل «click here» یا «free money» که بخشی از وابستگی بین کلمات را به مدل برمی‌گرداند.

نام پارامترها در Pipeline به شکل `<نام مرحله>__<نام پارامتر>` نوشته می‌شود.

</div>

In [ ]:
param_grid = {'vect__ngram_range': [(1, 1), (1, 2)],
              'clf__alpha': [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]}
grid = GridSearchCV(Pipeline([('vect', make_count_vectorizer()), ('clf', MultinomialNB())]),
                    param_grid, cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                    scoring='f1', n_jobs=-1)
t0 = time.time()
grid.fit(X_train_txt, y_train)
print(f'GridSearchCV finished in {time.time() - t0:.1f} s')
print('Best params :', grid.best_params_)
print(f'Best CV F1  : {grid.best_score_:.4f}')

res = pd.DataFrame(grid.cv_results_)
plt.figure(figsize=(10, 4.8))
for ngr, marker in [((1, 1), 'o'), ((1, 2), 's')]:
    sub = res[res['param_vect__ngram_range'] == ngr]
    plt.errorbar(sub['param_clf__alpha'].astype(float), sub['mean_test_score'], yerr=sub['std_test_score'],
                 marker=marker, capsize=4, label=f'ngram_range={ngr}')
plt.xscale('log'); plt.xlabel('alpha (Laplace smoothing, log scale)'); plt.ylabel('mean CV F1')
plt.title('GridSearchCV: effect of alpha and n-grams'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_txt)
print(f'\nTest set with the best model -> accuracy {accuracy_score(y_test, y_pred_best):.4f} | '
      f'precision {precision_score(y_test, y_pred_best):.4f} | recall {recall_score(y_test, y_pred_best):.4f} | '
      f'F1 {f1_score(y_test, y_pred_best):.4f}')

<div dir="rtl">

## ۱۰. کلمات مهم هر کلاس

یکی از بزرگ‌ترین مزایای Naive Bayes **تفسیرپذیری** آن است. مدل برای هر کلمه دو عدد یاد گرفته است: $\log P(w \mid \text{ham})$ و $\log P(w \mid \text{spam})$ که در `clf.feature_log_prob_` ذخیره شده‌اند. برای وضوح از مدل تک‌کلمه‌ای بخش ۵ (`nb_pipe`) استفاده می‌کنیم.

دو دیدگاه مختلف برای «کلمه مهم» وجود دارد:

### ۱۰-۱. پرتکرارترین کلمات هر کلاس: $P(w \mid c)$
کلماتی که بیشترین احتمال را در هر کلاس دارند. **مشکل:** کلمات عمومی ممکن است در هر دو کلاس پرتکرار باشند و قدرت تمایز نداشته باشند.

</div>

In [ ]:
log_prob = clf.feature_log_prob_           # shape (2, V): row 0 = ham, row 1 = spam
word_counts = clf.feature_count_          # raw counts of each word in each class (training set)
TOP = 20

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, c, name, color in [(axes[0], 0, 'HAM', 'tab:green'), (axes[1], 1, 'SPAM', 'tab:red')]:
    idx = np.argsort(log_prob[c])[::-1][:TOP]
    ax.barh(vocab[idx][::-1], np.exp(log_prob[c][idx])[::-1], color=color)
    ax.set_title(f'Top {TOP} most probable words in {name}:  P(word | {name.lower()})')
    ax.set_xlabel('probability')
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۱۰-۲. کلمات **متمایزکننده**: نسبت لگاریتمی (Log Probability Ratio)
معیار بهتر این است که ببینیم یک کلمه در کدام کلاس **نسبتاً** بیشتر است:
$$\text{LR}(w) = \log \frac{P(w \mid \text{spam})}{P(w \mid \text{ham})} = \log P(w \mid \text{spam}) - \log P(w \mid \text{ham})$$
- $\text{LR}(w) \gg 0$: هر بار دیدن این کلمه، شانس اسپم بودن را **$e^{\text{LR}}$ برابر** می‌کند.
- $\text{LR}(w) \ll 0$: نشانه قوی ایمیل سالم.
- $\text{LR}(w) \approx 0$: کلمه بی‌طرف است.

این دقیقاً همان مقداری است که هر تکرار کلمه به امتیاز نهایی $s(x)$ اضافه می‌کند. برای جلوگیری از غلبه کلمات خیلی نادر، فقط کلماتی را در نظر می‌گیریم که حداقل ۳۰ بار در داده آموزش آمده‌اند.

</div>

In [ ]:
log_ratio = log_prob[1] - log_prob[0]
total_count = word_counts.sum(axis=0)
MIN_COUNT = 30
freq_mask = total_count >= MIN_COUNT

words_df = pd.DataFrame({'word': vocab, 'count_ham': word_counts[0].astype(int), 'count_spam': word_counts[1].astype(int),
                         'P(w|ham)': np.exp(log_prob[0]), 'P(w|spam)': np.exp(log_prob[1]),
                         'log_ratio': log_ratio})[freq_mask]
top_spam = words_df.nlargest(25, 'log_ratio')
top_ham = words_df.nsmallest(25, 'log_ratio')

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
axes[0].barh(top_spam['word'][::-1], top_spam['log_ratio'][::-1], color='tab:red')
axes[0].set_title('Top 25 SPAM indicators\nlog P(w|spam) - log P(w|ham)'); axes[0].set_xlabel('log probability ratio')
axes[1].barh(top_ham['word'][::-1], -top_ham['log_ratio'][::-1], color='tab:green')
axes[1].set_title('Top 25 HAM indicators\nlog P(w|ham) - log P(w|spam)'); axes[1].set_xlabel('log probability ratio')
plt.tight_layout()
plt.show()

print('Most spam-indicative words:')
display(top_spam.head(15).round(6).set_index('word'))
print('Most ham-indicative words:')
display(top_ham.head(15).round(6).set_index('word'))

<div dir="rtl">

**تفسیر:**
- **کلمات اسپم** چند گروه‌اند: تبلیغات دارو (`pills`، `viagra`، `cialis`)، نرم‌افزار ارزان (`photoshop`، `macromedia`)، کالای تقلبی (`rolex`)، **تگ‌های HTML** (`nbsp`، `href`، `width` — چون ایمیل‌های اسپم اغلب HTML هستند) و **غلط‌های املایی عمدی** (مثل `wiil` یا `ooking`) که اسپمرها برای دور زدن فیلترها استفاده می‌کنند.
- **کلمات سالم** بیشتر اصطلاحات کاری Enron هستند: نام شرکت‌ها و افراد (`enron`، `dynegy`، `kaminski`، `fastow`)، اصطلاحات بازار انرژی و گاز (`mmbtu`، `ferc`، `hourahead`) و `ect` که بخشی از آدرس‌های ایمیل داخلی Enron است (مثل `hou/ect@ect`).

> **درس مهم (سوگیری داده):** مدل یاد گرفته که «ایمیل‌هایی که شبیه ایمیل‌های داخلی Enron هستند سالم‌اند». این روی همین دیتاست عالی کار می‌کند، اما برای صندوق ایمیل **شما** تعمیم نمی‌یابد! همیشه کلمات مهم مدل را بررسی کنید تا ببینید مدل واقعاً چه چیزی یاد گرفته است.

### ۱۰-۳. نمای کلی همه کلمات
هر نقطه یک کلمه است: محور افقی $\log P(w \mid \text{ham})$ و محور عمودی $\log P(w \mid \text{spam})$. کلمات روی قطر بی‌طرف‌اند؛ کلمات بالای قطر نشانه اسپم و پایین آن نشانه ایمیل سالم هستند. ستون‌های عمودی/افقی نقاط در لبه‌ها کلماتی‌اند که **هرگز** در یک کلاس دیده نشده‌اند و احتمالشان فقط از هموارسازی لاپلاس ($\alpha$) می‌آید.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))
lp_all = log_prob[:, freq_mask]
lr_all = log_ratio[freq_mask]
sc = ax.scatter(lp_all[0], lp_all[1], c=np.clip(lr_all, -6, 6), cmap='RdYlGn_r', s=8, alpha=0.7)
lims = [min(lp_all.min(), lp_all.min()) - 0.3, max(lp_all.max(), lp_all.max()) + 0.3]
ax.plot(lims, lims, 'k--', lw=1, label='neutral words (equal probability)')
frequent = words_df[words_df['count_ham'] + words_df['count_spam'] >= 300]      # label frequent words only
to_label = pd.concat([frequent.nlargest(8, 'log_ratio'), frequent.nsmallest(8, 'log_ratio')])
for k, (_, row) in enumerate(to_label.iterrows()):
    ax.annotate(row['word'], (np.log(row['P(w|ham)']), np.log(row['P(w|spam)'])), fontsize=9,
                xytext=(5, 12 * ((k % 4) - 1.5)), textcoords='offset points',
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
for w in ['free', 'money', 'click', 'meeting', 'thanks', 'price', 'offer', 'energy']:
    if w in vect.vocabulary_:
        j = vect.vocabulary_[w]
        ax.annotate(w, (log_prob[0, j], log_prob[1, j]), fontsize=10, weight='bold', color='navy',
                    xytext=(3, -10), textcoords='offset points')
ax.set_xlabel('log P(word | ham)'); ax.set_ylabel('log P(word | spam)')
ax.set_title(f'Every word with count >= {MIN_COUNT}: spam-ness vs ham-ness')
fig.colorbar(sc, ax=ax, label='log ratio (red = spam, green = ham)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

<div dir="rtl">

### ۱۰-۴. ابر کلمات (Word Cloud) — اختیاری
اگر کتابخانه `wordcloud` نصب باشد (`pip install wordcloud`)، اندازه هر کلمه متناسب با نسبت لگاریتمی آن رسم می‌شود.

</div>

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax, data, cmap, title in [(axes[0], words_df.nlargest(150, 'log_ratio'), 'Reds', 'SPAM words'),
                                  (axes[1], words_df.nsmallest(150, 'log_ratio'), 'Greens', 'HAM words')]:
        freqs = dict(zip(data['word'], np.abs(data['log_ratio'])))
        wc = WordCloud(width=800, height=450, background_color='white', colormap=cmap).generate_from_frequencies(freqs)
        ax.imshow(wc, interpolation='bilinear'); ax.axis('off'); ax.set_title(title, fontsize=16)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('wordcloud is not installed -> skipped.  Install with:  pip install wordcloud')

<div dir="rtl">

## ۱۱. توضیح پیش‌بینی برای یک ایمیل خاص

چون Naive Bayes یک مدل **خطی در فضای لگاریتم** است، امتیاز هر ایمیل دقیقاً به صورت جمع سهم تک‌تک کلمات تجزیه می‌شود:
$$s(x) = \underbrace{\log\frac{P(\text{spam})}{P(\text{ham})}}_{\text{prior}} + \sum_{w \in x} \underbrace{x_w \cdot \text{LR}(w)}_{\text{سهم کلمه } w}$$
پس می‌توانیم ببینیم **کدام کلمات** ایمیل را به سمت اسپم یا سالم هل داده‌اند. (بررسی می‌کنیم که مجموع سهم‌ها دقیقاً با امتیاز مدل برابر باشد.)

</div>

In [ ]:
def explain_email(text, top=12, show=True):
    x = vect.transform([text])
    idx, cnt = x.indices, x.data
    contrib = cnt * log_ratio[idx]
    prior = clf.class_log_prior_[1] - clf.class_log_prior_[0]
    total = prior + contrib.sum()
    lp = nb_pipe.predict_log_proba([text])[0]
    assert np.isclose(total, lp[1] - lp[0]), 'decomposition must match the model score'
    exp_df = pd.DataFrame({'word': vocab[idx], 'count': cnt, 'contribution': contrib})
    exp_df = exp_df.reindex(exp_df['contribution'].abs().sort_values(ascending=False).index).head(top)
    if show:
        short = re.sub(r'\s+', ' ', text)[:250]
        print(f'Email: {short} ...')
        print(f'prior = {prior:+.2f} | sum of word contributions = {contrib.sum():+.2f} | '
              f'score s(x) = {total:+.2f} -> {"SPAM" if total > 0 else "HAM"}')
    return exp_df, total


fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
# pick one correctly classified spam and ham email with a clear (but not extreme) score
spam_i = np.where((y_test == 1) & (score_test > 15) & (score_test < 80))[0][0]
ham_i = np.where((y_test == 0) & (score_test < -15) & (score_test > -80))[0][0]
examples = [(spam_i, 'a SPAM email'), (ham_i, 'a HAM email')]
for ax, (i, title) in zip(axes, examples):
    exp_df, total = explain_email(X_test_txt[i])
    print()
    exp_df = exp_df.iloc[::-1]
    ax.barh(exp_df['word'] + ' (x' + exp_df['count'].astype(str) + ')', exp_df['contribution'],
            color=np.where(exp_df['contribution'] > 0, 'tab:red', 'tab:green'))
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(f'{title}: score = {total:+.1f}\nred -> pushes to spam, green -> pushes to ham')
    ax.set_xlabel('contribution to the log-odds')
plt.tight_layout()
plt.show()

<div dir="rtl">

## ۱۲. تحلیل خطا

بررسی نمونه‌هایی که مدل اشتباه طبقه‌بندی کرده، ایده‌هایی برای بهبود می‌دهد (مثلاً افزودن ویژگی‌های جدید، پیش‌پردازش بهتر یا داده بیشتر).

</div>

In [ ]:
fp_idx = np.where((y_pred == 1) & (y_test == 0))[0]
fn_idx = np.where((y_pred == 0) & (y_test == 1))[0]
print(f'False positives (ham -> spam): {len(fp_idx)} | False negatives (spam -> ham): {len(fn_idx)}\n')
for title, idxs in [('FALSE POSITIVES (ham predicted as spam)', fp_idx[:4]), ('FALSE NEGATIVES (spam predicted as ham)', fn_idx[:4])]:
    print('=' * 20, title, '=' * 20)
    for i in idxs:
        print(f'[score {score_test[i]:+.1f}]', re.sub(r'\s+', ' ', X_test_txt[i])[:260], '...\n')

<div dir="rtl">

معمولاً خطاها شامل این موارد هستند:
- **ایمیل‌های سالم تبلیغاتی** (خبرنامه‌ها، پیشنهادهای تجاری واقعی) که واژگانی شبیه اسپم دارند؛
- **اسپم‌های کوتاه** یا اسپم‌هایی که عمداً کلمات «بی‌خطر» زیادی دارند تا فیلتر را فریب دهند (**Bayesian Poisoning**)؛
- اسپم‌هایی که متن اصلی آن‌ها در تصویر است و فقط چند کلمه متنی دارند.

</div>

<div dir="rtl">

## ۱۳. آزمایش با ایمیل‌های دلخواه
چند ایمیل ساختگی می‌نویسیم و پیش‌بینی مدل (و مدل بهینه GridSearch) را می‌بینیم. شما هم ایمیل‌های خود را امتحان کنید!

</div>

In [ ]:
my_emails = [
    'Congratulations! You have been selected to receive a FREE gift card. Click here to claim your prize now!',
    'Hi team, please find attached the agenda for tomorrow meeting. Let me know if you have any questions. Thanks',
    'Buy cheap viagra and cialis online, no prescription needed. Best prices, discreet shipping.',
    'Can you review the gas trading report before Friday? We need to discuss the pipeline capacity numbers.',
    'Limited time offer: refinance your mortgage today at the lowest rate. Unsubscribe here.',
    'Your invoice for the software license is attached. Please process the payment by the end of the month.',
]
rows = []
for e in my_emails:
    _, s = explain_email(e, show=False)
    rows.append({'email': e[:85] + '...', 'score s(x)': round(s, 2),
                 'P(spam)': nb_pipe.predict_proba([e])[0, 1],
                 'MultinomialNB': 'SPAM' if s > 0 else 'ham',
                 'best GridSearch model': 'SPAM' if best_model.predict([e])[0] == 1 else 'ham'})
pd.set_option('display.max_colwidth', 100)
display(pd.DataFrame(rows))

<div dir="rtl">

## ۱۴. جمع‌بندی

| موضوع | نکته کلیدی |
|---|---|
| ایده | قضیه بیز + فرض استقلال شرطی کلمات |
| پیاده‌سازی متن | `CountVectorizer` / `TfidfVectorizer` + `MultinomialNB` داخل یک `Pipeline` |
| هموارسازی | `alpha` از صفر شدن احتمال کلمات دیده‌نشده جلوگیری می‌کند |
| احتمالات | کالیبره نیستند (نزدیک ۰ و ۱)؛ برای آستانه از log-odds استفاده کنید |
| هزینه خطاها | در اسپم، FP (از دست رفتن ایمیل سالم) گران‌تر است ← آستانه را با داده اعتبارسنجی تنظیم کنید |
| تفسیرپذیری | `feature_log_prob_` ← کلمات مهم هر کلاس و توضیح پیش‌بینی هر ایمیل |
| مقایسه | NB بسیار سریع و قوی است؛ رگرسیون لجستیک با داده زیاد ممکن است کمی بهتر باشد |

### مزایا و معایب Naive Bayes
| مزایا | معایب |
|---|---|
| آموزش و پیش‌بینی **بسیار سریع** (یک بار عبور از داده) | فرض استقلال غیرواقعی |
| با داده کم خوب کار می‌کند | احتمالات کالیبره نیستند |
| ذاتاً چندکلاسه و قابل تفسیر | به سوگیری‌های واژگانی دیتاست حساس است |
| یادگیری افزایشی با `partial_fit` (مناسب فیلترهایی که مدام به‌روز می‌شوند) | ترتیب و معنای کلمات را نمی‌فهمد |

### تمرین‌ها
1. کلمات خاص Enron (مثل `enron`، `vince`، `kaminski`، `ect`، `hou`) را به فهرست Stop Words اضافه کنید و ببینید دقت و کلمات مهم چه تغییری می‌کنند.
2. ویژگی‌های اضافه مانند تعداد حروف بزرگ، تعداد علامت `!` و تعداد لینک‌ها را بسازید و با `ColumnTransformer` به مدل اضافه کنید.
3. با `CalibratedClassifierCV` احتمالات MultinomialNB را کالیبره کنید و هیستوگرام احتمالات را دوباره رسم کنید.
4. با `partial_fit` یک فیلتر «آنلاین» شبیه‌سازی کنید که ایمیل‌ها را به ترتیب تاریخ دریافت می‌کند.
5. مدل آموزش‌دیده روی Enron را روی دیتاست SMS Spam آزمایش کنید. نتیجه درباره تعمیم‌پذیری چه می‌گوید؟

</div>